In [1]:
from pathlib import Path
from datetime import timedelta

import earthaccess
import pandas as pd
import requests


PROJECT = Path(r"Z:\Projects\monsoon-postprocessing")

IMERG_FOLDER = (
    PROJECT
    / "data"
    / "raw"
    / "imerg_pilot"
)

GEFS_FOLDER = (
    PROJECT
    / "data"
    / "raw"
    / "gefs_pilot"
)

IMERG_FOLDER.mkdir(parents=True, exist_ok=True)
GEFS_FOLDER.mkdir(parents=True, exist_ok=True)

In [2]:
auth = earthaccess.login(
    strategy="interactive",
    persist=True
)

print("Authenticated:", auth.authenticated)

Enter your Earthdata Login username:  arayani
Enter your Earthdata password:  ········


Authenticated: True


In [3]:
imerg_results = earthaccess.search_data(
    short_name="GPM_3IMERGDF",
    version="07",
    temporal=(
        "2018-07-08",
        "2018-07-31"
    ),
    bounding_box=(
        68,
        6,
        98,
        38
    )
)

print("IMERG files found:", len(imerg_results))

IMERG files found: 24


In [4]:
downloaded_imerg = earthaccess.download(
    imerg_results,
    local_path=str(IMERG_FOLDER)
)

print("Download results:", len(downloaded_imerg))

QUEUEING TASKS | :   0%|          | 0/24 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/24 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/24 [00:00<?, ?it/s]

Download results: 24


In [5]:
imerg_files = sorted(
    IMERG_FOLDER.glob("*.nc4")
)

print("Total IMERG files:", len(imerg_files))

for file in imerg_files:
    print(file.name)

Total IMERG files: 31
3B-DAY.MS.MRG.3IMERG.20180701-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180702-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180703-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180704-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180705-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180706-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180707-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180708-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180709-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180710-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180711-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180712-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180713-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180714-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180715-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180716-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180717-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180718-S000000-E2359

In [6]:
GEFS_BASE_URL = (
    "https://noaa-gefs-retrospective.s3.amazonaws.com"
)


def build_gefs_url(observation_date):
    initialization_date = (
        observation_date - timedelta(days=1)
    )

    initialization = initialization_date.strftime(
        "%Y%m%d00"
    )

    year = initialization_date.strftime("%Y")

    filename = (
        f"apcp_sfc_{initialization}_c00.grib2"
    )

    url = (
        f"{GEFS_BASE_URL}/"
        f"GEFSv12/reforecast/{year}/"
        f"{initialization}/c00/Days:1-10/"
        f"{filename}"
    )

    return initialization_date, filename, url

In [7]:
def download_file(url, destination):
    """Download one file safely."""

    if destination.exists() and destination.stat().st_size > 0:
        print("Already available:", destination.name)
        return True

    temporary_file = destination.with_suffix(
        destination.suffix + ".part"
    )

    try:
        with requests.get(
            url,
            stream=True,
            timeout=(30, 300)
        ) as response:
            response.raise_for_status()

            expected_size = int(
                response.headers.get(
                    "content-length",
                    0
                )
            )

            with open(temporary_file, "wb") as output:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        output.write(chunk)

        downloaded_size = temporary_file.stat().st_size

        if expected_size and downloaded_size != expected_size:
            raise IOError("Incomplete download")

        temporary_file.replace(destination)

        print(
            f"Downloaded: {destination.name} "
            f"({downloaded_size / 1_000_000:.1f} MB)"
        )

        return True

    except Exception as error:
        print("Failed:", destination.name)
        print("Reason:", error)

        if temporary_file.exists():
            temporary_file.unlink()

        return False

In [8]:
observation_dates = pd.date_range(
    start="2018-07-08",
    end="2018-07-31",
    freq="D"
)

download_results = []


for observation_date in observation_dates:
    initialization_date, filename, url = (
        build_gefs_url(observation_date)
    )

    destination = GEFS_FOLDER / filename

    success = download_file(
        url,
        destination
    )

    download_results.append({
        "observation_date": observation_date.date(),
        "initialization_date": initialization_date.date(),
        "filename": filename,
        "success": success
    })


download_results_df = pd.DataFrame(
    download_results
)

download_results_df

Downloaded: apcp_sfc_2018070700_c00.grib2 (27.1 MB)
Downloaded: apcp_sfc_2018070800_c00.grib2 (26.9 MB)
Downloaded: apcp_sfc_2018070900_c00.grib2 (28.5 MB)
Downloaded: apcp_sfc_2018071000_c00.grib2 (27.3 MB)
Downloaded: apcp_sfc_2018071100_c00.grib2 (28.3 MB)
Downloaded: apcp_sfc_2018071200_c00.grib2 (29.5 MB)
Downloaded: apcp_sfc_2018071300_c00.grib2 (28.3 MB)
Downloaded: apcp_sfc_2018071400_c00.grib2 (27.9 MB)
Downloaded: apcp_sfc_2018071500_c00.grib2 (26.8 MB)
Downloaded: apcp_sfc_2018071600_c00.grib2 (25.9 MB)
Downloaded: apcp_sfc_2018071700_c00.grib2 (27.1 MB)
Downloaded: apcp_sfc_2018071800_c00.grib2 (26.4 MB)
Downloaded: apcp_sfc_2018071900_c00.grib2 (26.4 MB)
Downloaded: apcp_sfc_2018072000_c00.grib2 (26.5 MB)
Downloaded: apcp_sfc_2018072100_c00.grib2 (28.5 MB)
Downloaded: apcp_sfc_2018072200_c00.grib2 (26.6 MB)
Downloaded: apcp_sfc_2018072300_c00.grib2 (28.3 MB)
Downloaded: apcp_sfc_2018072400_c00.grib2 (27.5 MB)
Downloaded: apcp_sfc_2018072500_c00.grib2 (28.7 MB)
Downloaded: 

,observation_date,initialization_date,filename,success
0,2018-07-08,2018-07-07,apcp_sfc_2018070700_c00.grib2,True
1,2018-07-09,2018-07-08,apcp_sfc_2018070800_c00.grib2,True
2,2018-07-10,2018-07-09,apcp_sfc_2018070900_c00.grib2,True
3,2018-07-11,2018-07-10,apcp_sfc_2018071000_c00.grib2,True
4,2018-07-12,2018-07-11,apcp_sfc_2018071100_c00.grib2,True
5,2018-07-13,2018-07-12,apcp_sfc_2018071200_c00.grib2,True
6,2018-07-14,2018-07-13,apcp_sfc_2018071300_c00.grib2,True
7,2018-07-15,2018-07-14,apcp_sfc_2018071400_c00.grib2,True
8,2018-07-16,2018-07-15,apcp_sfc_2018071500_c00.grib2,True
9,2018-07-17,2018-07-16,apcp_sfc_2018071600_c00.grib2,True


In [9]:
gefs_files = sorted(
    GEFS_FOLDER.glob("*.grib2")
)

print("Total GEFS files:", len(gefs_files))

failed_downloads = download_results_df[
    ~download_results_df["success"]
]

print("Failed downloads:", len(failed_downloads))

Total GEFS files: 31
Failed downloads: 0
